## AI Content Understanding 

### Constants

In [38]:
import os
from dotenv import load_dotenv
load_dotenv()

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_VERSION = os.getenv("AZURE_AI_API_VERSION", "2025-05-01-preview")
AZURE_AI_API_KEY = os.getenv('AZURE_AI_API_KEY')

AZURE_TENANT_ID = os.getenv('AZURE_TENANT_ID')
AZURE_CLIENT_ID = os.getenv('AZURE_CLIENT_ID')
AZURE_CLIENT_SECRET = os.getenv('AZURE_CLIENT_SECRET')
AZUR_ENDPOINT = os.getenv('AZUR_ENDPOINT')
AZUR_OPENAI_KEY = os.getenv('AZUR_OPENAI_KEY')
# Constants
ANALYZER_ID = 'prebuilt-videoAnalyzer'
ANALYZER_SAMPLE_FILE = 'data/mcp.mp4'
INDEX_NAME = "video-content-understanding-index" 
SEARCH_KEY = os.getenv('SEARCH_KEY')
SEARCH_ENDPOINT = os.getenv('SEARCH_ENDPOINT')

VIDEO_CONTENT_UNDERSTANDING_KEY=os.getenv('VIDEO_CONTENT_UNDERSTANDING_KEY')
AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_EMBEDDING_ENDPOINT=os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_EMBEDDING_DEPLOYMENT_NAME=os.getenv('AZURE_EMBEDDING_DEPLOYMENT_NAME')

AZURE_CHAT_DEPLOYMENT_NAME=os.getenv('AZURE_CHAT_DEPLOYMENT_NAME')
AZURE_OPENAI_LLM_ENDPOINT=os.getenv('AZURE_OPENAI_LLM_ENDPOINT')



### Create Azure AI Content Understanding Client

In [2]:
import logging
import json
import os
import sys
import uuid
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv(find_dotenv())
logging.basicConfig(level=logging.INFO)

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_VERSION = os.getenv("AZURE_AI_API_VERSION", "2025-05-01-preview")

# Add the parent directory to the path to use shared modules
parent_dir = Path(Path.cwd()).parent
sys.path.append(str(parent_dir))
from python.content_understanding_client import AzureContentUnderstandingClient

credential = DefaultAzureCredential()
print(credential)
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    token_provider=token_provider,
    x_ms_useragent="azure-ai-content-understanding-python/content_extraction", # This header is used for sample usage telemetry, please comment out this line if you want to opt out.
)

INFO:azure.identity._credentials.environment:Environment is configured for ClientSecretCredential
INFO:azure.identity._credentials.managed_identity:ManagedIdentityCredential will use IMDS with client_id: 7e24b791-b55d-47a7-b21e-2041075056b7
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://login.microsoftonline.com/de2b428f-6cff-4dbf-b55a-db7c316662e8/v2.0/.well-known/openid-configuration'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.23.0 Python/3.12.7 (Windows-10-10.0.19045-SP0)'
No body was attached to the request


INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 200
Response headers:
    'Cache-Control': 'max-age=86400, private'
    'Content-Type': 'application/json; charset=utf-8'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Access-Control-Allow-Origin': 'REDACTED'
    'Access-Control-Allow-Methods': 'REDACTED'
    'P3P': 'REDACTED'
    'x-ms-request-id': '72372bcd-c9a2-4358-a0fc-3f78d3847f00'
    'x-ms-ests-server': 'REDACTED'
    'x-ms-srs': 'REDACTED'
    'Content-Security-Policy-Report-Only': 'REDACTED'
    'X-XSS-Protection': 'REDACTED'
    'Set-Cookie': 'REDACTED'
    'Date': 'Wed, 25 Jun 2025 02:01:41 GMT'
    'Content-Length': '1753'
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://login.microsoftonline.com/de2b428f-6cff-4dbf-b55a-db7c316662e8/oauth2/v2.0/token'
Request method: 'POST'
Request headers:
    'Accept': 'application/json'
    'x-client-sku': 'REDACTED'
    'x-client-ver': 'REDACTED'

In [ ]:
import json
import logging
import sys
import time
from collections.abc import Callable
from pathlib import Path
from typing import Any, cast
from dataclasses import dataclass

import requests


def upload_video_to_analyzer():
    settings = Settings(
        endpoint=AZURE_AI_ENDPOINT,
        api_version="2025-05-01-preview",
        # Either subscription_key or aad_token must be provided. Subscription Key is more prioritized.
        subscription_key=VIDEO_CONTENT_UNDERSTANDING_KEY,
        aad_token="AZURE_CONTENT_UNDERSTANDING_AAD_TOKEN",
        # Insert the analyzer name.
        analyzer_id=f"{ANALYZER_ID}",
        # Insert the supported file types of the analyzer.
        file_location="E:/Azure Video Indexer/data/mcp.mp4",
    )
    client = AzureContentUnderstandingClient(
        settings.endpoint,
        settings.api_version,
        subscription_key=settings.subscription_key,
        token_provider=settings.token_provider,
    )
    response = client.begin_analyze(settings.analyzer_id, settings.file_location)
    result = client.poll_result(
        response,
        timeout_seconds=60 * 60,
        polling_interval_seconds=1,
    )
    json.dump(result, sys.stdout, indent=2)
    video_id = result['id']

    return video_id


@dataclass(frozen=True, kw_only=True)
class Settings:
    endpoint: str
    api_version: str
    subscription_key: str | None = None
    aad_token: str | None = None
    analyzer_id: str
    file_location: str

    def __post_init__(self):
        key_not_provided = (
            not self.subscription_key
            or self.subscription_key == "AZURE_CONTENT_UNDERSTANDING_SUBSCRIPTION_KEY"
        )
        token_not_provided = (
            not self.aad_token
            or self.aad_token == "AZURE_CONTENT_UNDERSTANDING_AAD_TOKEN"
        )
        if key_not_provided and token_not_provided:
            raise ValueError(
                "Either 'subscription_key' or 'aad_token' must be provided"
            )

    @property
    def token_provider(self) -> Callable[[], str] | None:
        aad_token = self.aad_token
        if aad_token is None:
            return None

        return lambda: aad_token

class AzureContentUnderstandingClient:
    def __init__(
        self,
        endpoint: str,
        api_version: str,
        subscription_key: str | None = None,
        token_provider: Callable[[], str] | None = None,
        x_ms_useragent: str = "cu-sample-code",
    ) -> None:
        if not subscription_key and token_provider is None:
            raise ValueError(
                "Either subscription key or token provider must be provided"
            )
        if not api_version:
            raise ValueError("API version must be provided")
        if not endpoint:
            raise ValueError("Endpoint must be provided")

        self._endpoint: str = endpoint.rstrip("/")
        self._api_version: str = api_version
        self._logger: logging.Logger = logging.getLogger(__name__)
        self._logger.setLevel(logging.INFO)
        self._headers: dict[str, str] = self._get_headers(
            subscription_key, token_provider and token_provider(), x_ms_useragent
        )

    def begin_analyze(self, analyzer_id: str, file_location: str):
        """
        Begins the analysis of a file or URL using the specified analyzer.

        Args:
            analyzer_id (str): The ID of the analyzer to use.
            file_location (str): The path to the file or the URL to analyze.

        Returns:
            Response: The response from the analysis request.

        Raises:
            ValueError: If the file location is not a valid path or URL.
            HTTPError: If the HTTP request returned an unsuccessful status code.
        """
        if Path(file_location).exists():
            with open(file_location, "rb") as file:
                data = file.read()
            headers = {"Content-Type": "application/octet-stream"}
        elif "https://" in file_location or "http://" in file_location:
            data = {"url": file_location}
            headers = {"Content-Type": "application/json"}
        else:
            raise ValueError("File location must be a valid path or URL.")

        headers.update(self._headers)
        if isinstance(data, dict):
            response = requests.post(
                url=self._get_analyze_url(
                    self._endpoint, self._api_version, analyzer_id
                ),
                headers=headers,
                json=data,
            )
        else:
            response = requests.post(
                url=self._get_analyze_url(
                    self._endpoint, self._api_version, analyzer_id
                ),
                headers=headers,
                data=data,
            )

        response.raise_for_status()
        self._logger.info(
            f"Analyzing file {file_location} with analyzer: {analyzer_id}"
        )
        return response

    def poll_result(
        self,
        response: requests.Response,
        timeout_seconds: int = 120,
        polling_interval_seconds: int = 2,
    ) -> dict[str, Any]:  # pyright: ignore[reportExplicitAny]
        """
        Polls the result of an asynchronous operation until it completes or times out.

        Args:
            response (Response): The initial response object containing the operation location.
            timeout_seconds (int, optional): The maximum number of seconds to wait for the operation to complete. Defaults to 120.
            polling_interval_seconds (int, optional): The number of seconds to wait between polling attempts. Defaults to 2.

        Raises:
            ValueError: If the operation location is not found in the response headers.
            TimeoutError: If the operation does not complete within the specified timeout.
            RuntimeError: If the operation fails.

        Returns:
            dict: The JSON response of the completed operation if it succeeds.
        """
        operation_location = response.headers.get("operation-location", "")
        if not operation_location:
            raise ValueError("Operation location not found in response headers.")

        headers = {"Content-Type": "application/json"}
        headers.update(self._headers)

        start_time = time.time()
        while True:
            elapsed_time = time.time() - start_time
            self._logger.info(
                "Waiting for service response", extra={"elapsed": elapsed_time}
            )
            if elapsed_time > timeout_seconds:
                raise TimeoutError(
                    f"Operation timed out after {timeout_seconds:.2f} seconds."
                )

            response = requests.get(operation_location, headers=self._headers)
            response.raise_for_status()
            result = cast(dict[str, str], response.json())
            status = result.get("status", "").lower()
            if status == "succeeded":
                self._logger.info(
                    f"Request result is ready after {elapsed_time:.2f} seconds."
                )
                return response.json()  # pyright: ignore[reportAny]
            elif status == "failed":
                self._logger.error(f"Request failed. Reason: {response.json()}")
                raise RuntimeError("Request failed.")
            else:
                self._logger.info(
                    f"Request {operation_location.split('/')[-1].split('?')[0]} in progress ..."
                )
            time.sleep(polling_interval_seconds)

    def _get_analyze_url(self, endpoint: str, api_version: str, analyzer_id: str):
        return f"{endpoint}/contentunderstanding/analyzers/{analyzer_id}:analyze?api-version={api_version}&stringEncoding=utf16"

    def _get_headers(
        self, subscription_key: str | None, api_token: str | None, x_ms_useragent: str
    ) -> dict[str, str]:
        """Returns the headers for the HTTP requests.
        Args:
            subscription_key (str): The subscription key for the service.
            api_token (str): The API token for the service.
            enable_face_identification (bool): A flag to enable face identification.
        Returns:
            dict: A dictionary containing the headers for the HTTP requests.
        """
        headers = (
            {"Ocp-Apim-Subscription-Key": subscription_key}
            if subscription_key
            else {"Authorization": f"Bearer {api_token}"}
        )
        headers["x-ms-useragent"] = x_ms_useragent
        return headers

if __name__ == "__main__":
    video_id = upload_video_to_analyzer()


In [5]:
import requests

def get_analyzer_result(result_id):
    url = f"{AZURE_AI_ENDPOINT}/contentunderstanding/analyzerResults/{result_id}?api-version=2025-05-01-preview"
    headers = {
        "Ocp-Apim-Subscription-Key": VIDEO_CONTENT_UNDERSTANDING_KEY
    }

    response = requests.get(url, headers=headers)

    print(f"Status Code: {response.status_code}")
    print(f"Response Headers: {response.headers}")
    print(f"Response Body: {response.text}")

    return response


In [49]:
index_name = INDEX_NAME
'''
No need to run this cell again.
This code block was only used to create the Azure Cognitive Search index.
It has been commented out to prevent accidental re-execution.
'''

from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes.models import (
    SearchField,
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmMetric,
    ExhaustiveKnnAlgorithmConfiguration,
    ExhaustiveKnnParameters,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    SearchIndex,
    AzureOpenAIParameters,
)
admin_key = SEARCH_KEY
index_name = INDEX_NAME

client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(admin_key)
)

# Define the index fields
fields = [
    SimpleField(name="videoId", type="Edm.String", key=True),
    SearchableField(name="transcript", type="Edm.String", vector_search_embedding_configuration="myOpenAI"),
    SearchableField(name="segment_analysis", type="Edm.String", vector_search_embedding_configuration="myOpenAI"),
    SearchField(
        name="SegmentAnalysisVector", 
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single), vector_search_dimensions=1536, vector_search_profile_name="myHnswProfile"
    ),
    SearchField(
        name="transcriptVector", 
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single), vector_search_dimensions=1536, vector_search_profile_name="myHnswProfile"
    ),
]

# Vectore search profile 
vector_search = VectorSearch(  
    algorithms=[  
        HnswAlgorithmConfiguration(  
            name="myHnsw",  
            parameters=HnswParameters(  
                m=4,  
                ef_construction=400,  
                ef_search=500,  
                metric=VectorSearchAlgorithmMetric.COSINE,  
            ),  
        ),  
        ExhaustiveKnnAlgorithmConfiguration(  
            name="myExhaustiveKnn",  
            parameters=ExhaustiveKnnParameters(  
                metric=VectorSearchAlgorithmMetric.COSINE,  
            ),  
        ),  
    ],  
    profiles=[  
        VectorSearchProfile(  
            name="myHnswProfile",  
            algorithm_configuration_name="myHnsw",  
            vectorizer="myOpenAI",  
        )
    ],  
    vectorizers=[  
        AzureOpenAIVectorizer(  
            name="myOpenAI",  
            kind="azureOpenAI",  
            azure_open_ai_parameters=AzureOpenAIParameters(  
                resource_uri=AZUR_ENDPOINT,  
                deployment_id="text-embedding-ada-002dev",  
                api_key=AZUR_OPENAI_KEY,  
            ),  
        ),  
    ],
)  

# Create the search index
index = SearchIndex(
    name=index_name,  # Make sure index_name is defined
    fields=fields,
    vector_search=vector_search,
)

# Create the index/
try:
    result = client.create_or_update_index(index)
    print(f"Index '{index_name}' created successfully")
except Exception as e:
    print(f"Error creating index: {e}")


Index 'video-content-understanding-index' created successfully


In [ ]:
response = get_analyzer_result(video_id)


In [50]:
import json
import uuid
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

Azure_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2023-05-15",  # Use your deployed API version
    azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT  # e.g., "https://<your-resource-name>.openai.azure.com/"
)

def create_embedding(text: str) -> list[float]:
    response = Azure_client.embeddings.create(
        input=text,
        model=AZURE_EMBEDDING_DEPLOYMENT_NAME
    )
    return response.data[0].embedding

response = get_analyzer_result(video_id)
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=index_name,
    credential=AzureKeyCredential(admin_key)
)
segments = response.json()['result']['contents'][0]['fields']['Segments']['valueArray']
descriptions = [
    segment['valueObject']['Description']['valueString']
    for segment in segments
]
description_text = "\n\n".join(descriptions)

phrases = response.json()['result']['contents'][0].get('transcriptPhrases', [])
transcript_text = " ".join([phrase['text'] for phrase in phrases])

combined_text = f"Segment Summaries:\n{description_text}\n\nFull Transcript:\n{transcript_text}"


doc = { 
    'videoId': video_id,
    'transcript': transcript_text,
    'transcriptVector' : create_embedding(combined_text) ,
    'segment_analysis': description_text,
    'SegmentAnalysisVector' : create_embedding(description_text) 
}

search_client.upload_documents(documents=[doc])

Status Code: 200
Response Headers: {'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json', 'request-id': '6ebf1eb7-c99c-45b8-a64f-595518daacad', 'x-ms-request-id': '6ebf1eb7-c99c-45b8-a64f-595518daacad', 'api-supported-versions': '2024-12-01-preview,2025-05-01-preview', 'x-envoy-upstream-service-time': '61', 'apim-request-id': '6ebf1eb7-c99c-45b8-a64f-595518daacad', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'Australia East', 'Date': 'Thu, 26 Jun 2025 08:11:42 GMT'}
Response Body: {"id":"cc0643e2-8be8-4c4b-8841-6fae29e718c6","status":"Succeeded","result":{"analyzerId":"prebuilt-videoAnalyzer","apiVersion":"2025-05-01-preview","createdAt":"2025-06-26T06:06:56Z","stringEncoding":"utf8","warnings":[],"contents":[{"markdown":"# Video: 00:00.000 => 03:45.792\nWidth: 1280\nHeight: 720\n\n## Segment 1: 00:00.000 => 00:56.463\nThe video begins with an introduction to MCP, the Model Context Protoc

In [ ]:
def IndexSearch(question_vector):
    """
    Function information:
        Info: Do vector search from exechat index  
    """
    try:
        
        # Headers and Params value 
        headers = {'Content-Type': 'application/json', 'api-key': SEARCH_KEY }
        params = {'api-version': '2023-11-01'}

        index_payload = {
            "select": "transcript,segment_analysis",
            "count": True,
            "vectorQueries": [
                {
                    "vector": question_vector,
                    "fields": "transcriptVector",
                    "k": 3,
                    "kind": "vector"
                },
                {
                    "vector": question_vector,
                    "fields": "SegmentAnalysisVector",
                    "k": 3,
                    "kind": "vector"
                }
            ]
        }

        Request = requests.post(SEARCH_ENDPOINT + "/indexes/" + INDEX_NAME +"/docs/search", 
            data=json.dumps(index_payload), headers=headers, params=params)
        Response = Request.json()
        return True, Response['value']
    except Exception as e:
        print(e)
        return False, None

In [ ]:
from openai import AzureOpenAI

def ask_question_with_context(user_question):
    user_question_vector = create_embedding(user_question)
    context = IndexSearch(user_question_vector)
    print(context)

    # Azure OpenAI client
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version="2024-12-01-preview",
        azure_endpoint=AZURE_OPENAI_LLM_ENDPOINT
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # This must be your deployed name (e.g., "gpt-4-deployment")
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use ONLY the provided context to answer in detail the user's question. "
                    "Do not rely on any outside knowledge or make assumptions beyond the context. "
                    "If the answer cannot be found in the context, reply with: 'The context does not contain that information.'"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}"
            },
            {
                "role": "user",
                "content": f"Question: {user_question}"
            }
        ],
        temperature=0.3,
        max_tokens=1024
    )
    return response.choices[0].message.content


In [55]:
ask_question_with_context("What is MCP?")

(True, [{'@search.score': 0.8582021, 'segment_analysis': "The video begins with an introduction to MCP, the Model Context Protocol, which is an open-source standard for connecting AI agents to data sources like databases or APIs. The speaker, identified as Roy Derks from IBM Software, explains the components of MCP, including the host, client, and server. He describes the MCP host as an application that can include multiple clients, such as a chat app or a code assistant in an IDE. The host connects to one or more MCP servers using the MCP protocol. Throughout the segment, the speaker uses a transparent board to illustrate the structure of MCP, drawing boxes and labels to represent the host, clients, and servers.\n\nThe video segment begins with a person explaining the Model Content Protocol (MCP) using a diagram on a blackboard. The person draws and labels components such as 'mcp host', 'mcp protocol', and 'mcp server'. They explain how the MCP host or client connects to the MCP serve

'MCP stands for Model Context Protocol. It is an open-source standard designed for connecting AI agents to various data sources, such as databases or APIs. The protocol involves several components, including the MCP host, client, and server. The MCP host is an application that can support multiple clients, like a chat application or a code assistant within an Integrated Development Environment (IDE). The host connects to one or more MCP servers using the MCP protocol, which facilitates the interaction between clients and data sources, enabling tasks such as retrieving information from databases or APIs.'